In [3]:
import os
from brevitas.export import export_qonnx
import onnx
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import gc
from brevitas.nn import QuantConv2d, QuantLinear, QuantReLU, TruncAvgPool2d
from brevitas.quant import Int32Bias
from configparser import ConfigParser
from torch.nn import Sequential
from brevitas.core.restrict_val import RestrictValueType
from brevitas.quant import Int8ActPerTensorFloat
from brevitas.quant import Int8WeightPerTensorFloat
from brevitas.quant import Uint8ActPerTensorFloat
from tqdm import tqdm
import copy
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from brevitas.export import export_qonnx
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.util.cleanup import cleanup as qonnx_cleanup
from qonnx.core.datatype import DataType
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from finn.util.visualization import showSrc
from qonnx.transformation.general import GiveReadableTensorNames, GiveUniqueNodeNames,RemoveStaticGraphInputs
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.fold_constants import FoldConstants
import finn.transformation.streamline.absorb as absorb
import finn.transformation.streamline.reorder as reorder
from finn.transformation.streamline import Streamline
from qonnx.transformation.double_to_single_float import DoubleToSingleFloat
from qonnx.transformation.change_datalayout import ChangeDataLayoutQuantAvgPool2d
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from finn.transformation.streamline.collapse_repeated import CollapseRepeatedMul
from qonnx.transformation.remove import RemoveIdentityOps
from finn.transformation.streamline.round_thresholds import RoundAndClipThresholds
from qonnx.transformation.lower_convs_to_matmul import LowerConvsToMatMul
import finn.transformation.fpgadataflow.convert_to_hw_layers as to_hw
from finn.transformation.fpgadataflow.create_dataflow_partition import CreateDataflowPartition
import finn.builder.build_dataflow as build
import finn.builder.build_dataflow_config as build_cfg
import shutil
from finn.transformation.streamline.reorder import MoveScalarLinearPastInvariants, MakeMaxPoolNHWC
from qonnx.transformation.insert_topk import InsertTopK
from finn.transformation.streamline.absorb import AbsorbScalarMulAddIntoTopK,AbsorbAddIntoMultiThreshold,AbsorbMulIntoMultiThreshold
from torchvision.models import vgg16, VGG16_Weights
from sklearn.metrics import accuracy_score, classification_report, cohen_kappa_score, confusion_matrix
from tqdm import tqdm
from brevitas import config
import warnings

In [2]:
pip install seaborn

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 KB 2.1 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


## Dataset

In [2]:
BASE_PATH = 'DDR-dataset/DR_grading'
TRAIN_PATH = os.path.join(BASE_PATH, 'train')
VALID_PATH = os.path.join(BASE_PATH, 'valid')
TEST_PATH = os.path.join(BASE_PATH, 'test')

In [3]:
def load_labels(file_path):
    labels = pd.read_csv(file_path, sep=' ', header=None, names=['filename', 'label'])
    return labels

train_labels = load_labels(os.path.join(BASE_PATH, 'train.txt'))
valid_labels = load_labels(os.path.join(BASE_PATH, 'valid.txt'))
test_labels = load_labels(os.path.join(BASE_PATH, 'test.txt'))

In [4]:
class CustomDataset(Dataset):
    def __init__(self, labels, dir_path, transform=None):
        self.labels = labels
        self.dir_path = dir_path
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img_name = os.path.join(self.dir_path, self.labels.iloc[idx, 0])
        image = Image.open(img_name).convert('RGB')
        label = int(self.labels.iloc[idx, 1])
        if self.transform:
            image = self.transform(image)
        return image, label

In [5]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(20),
    transforms.RandomHorizontalFlip(),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [6]:
BATCH_SIZE = 16  # Tamanho do batch

train_dataset = CustomDataset(train_labels, TRAIN_PATH, transform=train_transform)
valid_dataset = CustomDataset(valid_labels, VALID_PATH, transform=test_transform)
test_dataset = CustomDataset(test_labels, TEST_PATH, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

## Definindo modelo não quantizado

In [7]:
weights = VGG16_Weights.DEFAULT
model = vgg16(weights=weights)

num_ftrs = model.classifier[0].in_features  # 512*7*7
model.classifier = nn.Sequential(
    nn.Linear(num_ftrs, 256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, 6)
)

In [8]:
model

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

## Definindo modelo quantizado

In [12]:
# Copyright (C) 2023, Advanced Micro Devices, Inc. All rights reserved.
# SPDX-License-Identifier: BSD-3-Clause

class CommonIntWeightPerTensorQuant(Int8WeightPerTensorFloat):
    scaling_min_val = 2e-16
    bit_width = None

class CommonIntWeightPerChannelQuant(CommonIntWeightPerTensorQuant):
    scaling_per_output_channel = True

class CommonIntActQuant(Int8ActPerTensorFloat):
    scaling_min_val = 2e-16
    bit_width = None
    restrict_scaling_type = RestrictValueType.LOG_FP

class CommonUintActQuant(Uint8ActPerTensorFloat):
    scaling_min_val = 2e-16
    bit_width = None
    restrict_scaling_type = RestrictValueType.LOG_FP

In [13]:
class QuantVGG16(nn.Module):
    def __init__(self, weight_bit_width, act_bit_width, num_classes=6):
        super(QuantVGG16, self).__init__()
        
        self.features = nn.Sequential(
            # Bloco 1
            QuantConv2d(3, 64, kernel_size=3, stride=1, padding=1,groups=1,
                        bias=True, weight_bit_width=weight_bit_width,
                        weight_quant=CommonIntWeightPerChannelQuant),
            QuantReLU(act_quant=CommonUintActQuant,
                      bit_width=act_bit_width, return_quant_tensor=True),
            
            QuantConv2d(64, 64, kernel_size=3, stride=1, padding=1,groups=1,
                        bias=True, weight_bit_width=weight_bit_width,
                        weight_quant=CommonIntWeightPerChannelQuant),
            QuantReLU(act_quant=CommonUintActQuant,
                      bit_width=act_bit_width, return_quant_tensor=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Bloco 2
            QuantConv2d(64, 128, kernel_size=3, stride=1, padding=1,groups=1,
                        bias=True, weight_bit_width=weight_bit_width,
                        weight_quant=CommonIntWeightPerChannelQuant),
            QuantReLU(act_quant=CommonUintActQuant,
                      bit_width=act_bit_width, return_quant_tensor=True),
            
            QuantConv2d(128, 128, kernel_size=3, stride=1, padding=1,groups=1,
                        bias=True, weight_bit_width=weight_bit_width,
                        weight_quant=CommonIntWeightPerChannelQuant),
            QuantReLU(act_quant=CommonUintActQuant,
                      bit_width=act_bit_width, return_quant_tensor=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Bloco 3
            QuantConv2d(128, 256, kernel_size=3, stride=1, padding=1,groups=1,
                        bias=True, weight_bit_width=weight_bit_width,
                        weight_quant=CommonIntWeightPerChannelQuant),
            QuantReLU(act_quant=CommonUintActQuant,
                      bit_width=act_bit_width, return_quant_tensor=True),
            
            QuantConv2d(256, 256, kernel_size=3, stride=1, padding=1,groups=1,
                        bias=True, weight_bit_width=weight_bit_width,
                        weight_quant=CommonIntWeightPerChannelQuant),
            QuantReLU(act_quant=CommonUintActQuant,
                      bit_width=act_bit_width, return_quant_tensor=True),
            
            QuantConv2d(256, 256, kernel_size=3, stride=1, padding=1,groups=1,
                        bias=True, weight_bit_width=weight_bit_width,
                        weight_quant=CommonIntWeightPerChannelQuant),
            QuantReLU(act_quant=CommonUintActQuant,
                      bit_width=act_bit_width, return_quant_tensor=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Bloco 4
            QuantConv2d(256, 512, kernel_size=3, stride=1, padding=1,groups=1,
                        bias=True, weight_bit_width=weight_bit_width,
                        weight_quant=CommonIntWeightPerChannelQuant),
            QuantReLU(act_quant=CommonUintActQuant,
                      bit_width=act_bit_width, return_quant_tensor=True),
            
            QuantConv2d(512, 512, kernel_size=3, stride=1, padding=1,groups=1,
                        bias=True, weight_bit_width=weight_bit_width,
                        weight_quant=CommonIntWeightPerChannelQuant),
            QuantReLU(act_quant=CommonUintActQuant,
                      bit_width=act_bit_width, return_quant_tensor=True),
            
            QuantConv2d(512, 512, kernel_size=3, stride=1, padding=1,groups=1,
                        bias=True, weight_bit_width=weight_bit_width,
                        weight_quant=CommonIntWeightPerChannelQuant),
            QuantReLU(act_quant=CommonUintActQuant,
                      bit_width=act_bit_width, return_quant_tensor=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Bloco 5
            QuantConv2d(512, 512, kernel_size=3, stride=1, padding=1,groups=1,
                        bias=True, weight_bit_width=weight_bit_width,
                        weight_quant=CommonIntWeightPerChannelQuant),
            QuantReLU(act_quant=CommonUintActQuant,
                      bit_width=act_bit_width, return_quant_tensor=True),
            
            QuantConv2d(512, 512, kernel_size=3, stride=1, padding=1,groups=1,
                        bias=True, weight_bit_width=weight_bit_width,
                        weight_quant=CommonIntWeightPerChannelQuant),
            QuantReLU(act_quant=CommonUintActQuant,
                      bit_width=act_bit_width, return_quant_tensor=True),
            
            QuantConv2d(512, 512, kernel_size=3, stride=1, padding=1,groups=1,
                        bias=True, weight_bit_width=weight_bit_width,
                        weight_quant=CommonIntWeightPerChannelQuant),
            QuantReLU(act_quant=CommonUintActQuant,
                      bit_width=act_bit_width, return_quant_tensor=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
        )

        #self.avgpool = TruncAvgPool2d(kernel_size=(7,7), stride=1, bit_width=act_bit_width)

        self.classifier = nn.Sequential(
            QuantLinear(512*7*7, 
                        256, 
                        bias=True, 
                        weight_quant=CommonIntWeightPerChannelQuant,
                        weight_bit_width=weight_bit_width),
            QuantReLU(act_quant=CommonUintActQuant, 
                      bit_width=act_bit_width, 
                      return_quant_tensor=True),
            nn.Dropout(0.4),
            QuantLinear(256, 
                        num_classes, 
                        bias=True, 
                        weight_quant=CommonIntWeightPerChannelQuant,
                        weight_bit_width=weight_bit_width),
        )
        self._initialize_weights()

    def forward(self, x):
        x = self.features(x)
        #x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

## Treinando o modelo não quantizado 

In [17]:
def train_model(model, criterion, optimizer, train_loader, valid_loader, model_name, num_epochs=20):
    best_model_wts = model.state_dict()
    best_acc = 0.0
    
    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)
        
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                data_loader = train_loader
            else:
                model.eval()
                data_loader = valid_loader
                
            running_loss = 0.0
            running_corrects = 0
            all_preds = []
            all_labels = []
            
            for inputs, labels in data_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)
                
                optimizer.zero_grad()
                
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
            
            epoch_loss = running_loss / len(data_loader.dataset)
            epoch_acc = running_corrects.double() / len(data_loader.dataset)
            
            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
            torch.save(model.state_dict(), f'{model_name}_epoch{epoch+1}.pth')
            
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = model.state_dict()
                torch.save(best_model_wts, model_name)
    
    print(f'Best val Acc: {best_acc:4f}')
    model.load_state_dict(best_model_wts)
    return model

In [13]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

Congelando o Backbone para as primeiras épocas.

In [14]:
for param in model.features.parameters():
    param.requires_grad = False

In [15]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.classifier.parameters(),lr=0.001)

Treinando a cabeça do modelo por 5 épocas.

In [18]:
model = train_model(model, criterion, optimizer, train_loader, valid_loader,"best_vgg16.pth", num_epochs=5)

Epoch 1/5
----------
train Loss: 1.1571 Acc: 0.5625
val Loss: 0.8871 Acc: 0.6945
Epoch 2/5
----------
train Loss: 1.0279 Acc: 0.6107
val Loss: 0.8530 Acc: 0.6868
Epoch 3/5
----------
train Loss: 1.0149 Acc: 0.6127
val Loss: 0.8841 Acc: 0.6718
Epoch 4/5
----------
train Loss: 0.9819 Acc: 0.6244
val Loss: 0.8423 Acc: 0.6758
Epoch 5/5
----------
train Loss: 0.9745 Acc: 0.6344
val Loss: 0.8003 Acc: 0.6992
Best val Acc: 0.699232


"Descongelando" o Backbone.

In [19]:
for param in model.features.parameters():
    param.requires_grad = True

In [20]:
optimizer = torch.optim.Adam(model.parameters(),lr=0.0001)             

In [21]:
model = train_model(model, criterion, optimizer, train_loader, valid_loader,"best_vgg16.pth", num_epochs=20)

Epoch 1/20
----------
train Loss: 1.0317 Acc: 0.6173
val Loss: 0.8518 Acc: 0.6831
Epoch 2/20
----------
train Loss: 0.9099 Acc: 0.6657
val Loss: 0.8333 Acc: 0.7051
Epoch 3/20
----------
train Loss: 0.8557 Acc: 0.6872
val Loss: 0.6963 Acc: 0.7655
Epoch 4/20
----------
train Loss: 0.8083 Acc: 0.7122
val Loss: 0.6557 Acc: 0.7885
Epoch 5/20
----------
train Loss: 0.7628 Acc: 0.7222
val Loss: 0.6593 Acc: 0.7896
Epoch 6/20
----------
train Loss: 0.7414 Acc: 0.7320
val Loss: 0.7986 Acc: 0.7541
Epoch 7/20
----------
train Loss: 0.7187 Acc: 0.7481
val Loss: 0.7483 Acc: 0.7468
Epoch 8/20
----------
train Loss: 0.7032 Acc: 0.7486
val Loss: 0.6379 Acc: 0.7958
Epoch 9/20
----------
train Loss: 0.6865 Acc: 0.7546
val Loss: 0.5915 Acc: 0.8149
Epoch 10/20
----------
train Loss: 0.6599 Acc: 0.7633
val Loss: 0.5809 Acc: 0.8244
Epoch 11/20
----------
train Loss: 0.6365 Acc: 0.7725
val Loss: 0.6393 Acc: 0.7951
Epoch 12/20
----------
train Loss: 0.6268 Acc: 0.7785
val Loss: 0.6240 Acc: 0.8013
Epoch 13/20
-

Avaliando o modelo

## Avaliando o Modelo Não Quantizado

In [7]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.load_state_dict(torch.load("best_vgg16.pth", map_location=device))
model = model.to(device)
model.eval()

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [16]:
def calculate_metrics(loader, model, device):
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=[str(i) for i in range(6)], output_dict=True)
    kappa = cohen_kappa_score(all_labels, all_preds)
    
    per_class_accuracy = [report[str(i)]['precision'] for i in range(6)]
    mean_accuracy = sum(per_class_accuracy) / len(per_class_accuracy)

    return accuracy, mean_accuracy, kappa, classification_report(all_labels, all_preds, target_names=[str(i) for i in range(6)]), confusion_matrix(all_labels, all_preds)

In [17]:
def plot_confusion_matrix(cm, title, filename):
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[str(i) for i in range(6)], yticklabels=[str(i) for i in range(6)])
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title(title)
    plt.savefig(filename)
    plt.close()

In [18]:
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
print("Validation Set Metrics:")
valid_accuracy, valid_mean_accuracy, valid_kappa, valid_report, valid_conf_matrix = calculate_metrics(valid_loader, model, device)
print(f"Validation OA: {valid_accuracy:.4f}")
print(f"Validation AA: {valid_mean_accuracy:.4f}")
print(f"Validation Kappa: {valid_kappa:.4f}")
print("Validation Classification Report:")
print(valid_report)
print("Validation Confusion Matrix:")
print(valid_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de validação
plot_confusion_matrix(valid_conf_matrix, 'Validation Confusion Matrix', 'validation_confusion_matrix_vgg16.png')

Validation Set Metrics:


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classif

Validation OA: 0.8427
Validation AA: 0.6603
Validation Kappa: 0.7542
Validation Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.97      0.91      1253
           1       0.00      0.00      0.00       126
           2       0.82      0.84      0.83       895
           3       0.47      0.40      0.44        47
           4       0.86      0.65      0.74       182
           5       0.96      0.87      0.91       230

    accuracy                           0.84      2733
   macro avg       0.66      0.62      0.64      2733
weighted avg       0.80      0.84      0.82      2733

Validation Confusion Matrix:
[[1217    0   36    0    0    0]
 [  89    0   35    0    1    1]
 [ 116    0  748   18    7    6]
 [   0    0   24   19    4    0]
 [   4    0   55    3  119    1]
 [   8    0   14    0    8  200]]


In [20]:
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
print("Test Set Metrics:")
test_accuracy, test_mean_accuracy, test_kappa, test_report, test_conf_matrix = calculate_metrics(test_loader, model, device,zero_division=0)
print(f"Test OA: {test_accuracy:.4f}")
print(f"Test AA: {test_mean_accuracy:.4f}")
print(f"Test Kappa: {test_kappa:.4f}")
print("Test Classification Report:")
print(test_report)
print("Test Confusion Matrix:")
print(test_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de teste
plot_confusion_matrix(test_conf_matrix, 'Test Confusion Matrix', 'test_confusion_matrix_vgg16.png')

Test Set Metrics:


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classif

Test OA: 0.7413
Test AA: 0.6025
Test Kappa: 0.5890
Test Classification Report:
              precision    recall  f1-score   support

           0       0.73      0.97      0.84      1880
           1       0.00      0.00      0.00       189
           2       0.74      0.55      0.63      1344
           3       0.52      0.18      0.27        71
           4       0.85      0.48      0.62       275
           5       0.78      0.93      0.85       346

    accuracy                           0.74      4105
   macro avg       0.60      0.52      0.53      4105
weighted avg       0.71      0.74      0.71      4105

Test Confusion Matrix:
[[1832    0   48    0    0    0]
 [ 135    0   52    0    0    2]
 [ 527    0  742    5    8   62]
 [   2    0   52   13    4    0]
 [   7    0   99    7  133   29]
 [   1    0   10    0   12  323]]


## Treinando o modelos quantizados

In [11]:
model.load_state_dict(torch.load("best_vgg16.pth"))

<All keys matched successfully>

In [12]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [13]:
def train_qat(model, criterion, optimizer, train_loader, valid_loader, model_name, num_epochs=10):
    best_model_wts = model.state_dict()
    best_acc = 0.0
    
    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)
        
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                data_loader = train_loader
            else:
                model.eval()
                data_loader = valid_loader
                
            running_loss = 0.0
            running_corrects = 0
            all_preds = []
            all_labels = []
            
            for inputs, labels in data_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)
                
                optimizer.zero_grad()
                
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
            
            epoch_loss = running_loss / len(data_loader.dataset)
            epoch_acc = running_corrects.double() / len(data_loader.dataset)
            
            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
            torch.save(model.state_dict(), f'{model_name}_epoch{epoch+1}.pth')
            
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = model.state_dict()
                torch.save(best_model_wts, model_name)
    
    print(f'Best val Acc: {best_acc:4f}')
    model.load_state_dict(best_model_wts)
    return model

In [14]:
BATCH_SIZE = 8  # Tamanho do batch reduzido para caber na GPU

train_dataset = CustomDataset(train_labels, TRAIN_PATH, transform=train_transform)
valid_dataset = CustomDataset(valid_labels, VALID_PATH, transform=test_transform)
test_dataset = CustomDataset(test_labels, TEST_PATH, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

Loop de treinamento com modelos w8a8, w8a4, w4a8, w4a4

In [15]:
for weight, act in list(zip([8,8,4,4],[8,4,8,4])):
    print(f"QAT para pesos de {weight} bits e ativações de {act} bits;")
    model_name = f'qat_vgg16_w{weight}_a{act}.pth'
    
    quant_model = QuantVGG16(weight,act)# definindo modelo
    quant_model.load_state_dict(model.state_dict(), strict=False) # copiando pesos
    quant_model.to(device) #movendo para a gpu

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(quant_model.parameters(),lr=0.0001)

    quant_model = train_qat(quant_model,criterion,optimizer,train_loader,valid_loader,model_name)

QAT para pesos de 8 bits e ativações de 8 bits;
Epoch 1/10
----------


/usr/local/lib/python3.10/dist-packages/torch/_tensor.py:1255: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at ../c10/core/TensorImpl.h:1758.)
  return super(Tensor, self).rename(names)
/tmp/ipykernel_4001/2241107938.py:119: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:350.)
  x = torch.flatten(x, 1)


train Loss: 0.5837 Acc: 0.7955
val Loss: 0.6482 Acc: 0.8002
Epoch 2/10
----------
train Loss: 0.5643 Acc: 0.8023
val Loss: 0.5274 Acc: 0.8390
Epoch 3/10
----------
train Loss: 0.5650 Acc: 0.7991
val Loss: 0.6386 Acc: 0.8178
Epoch 4/10
----------
train Loss: 0.5513 Acc: 0.8045
val Loss: 0.7538 Acc: 0.7816
Epoch 5/10
----------
train Loss: 0.5454 Acc: 0.8031
val Loss: 0.5354 Acc: 0.8401
Epoch 6/10
----------
train Loss: 0.5438 Acc: 0.8101
val Loss: 0.5586 Acc: 0.8321
Epoch 7/10
----------
train Loss: 0.5176 Acc: 0.8186
val Loss: 0.7596 Acc: 0.7827
Epoch 8/10
----------
train Loss: 0.5164 Acc: 0.8114
val Loss: 0.6990 Acc: 0.7947
Epoch 9/10
----------
train Loss: 0.5074 Acc: 0.8219
val Loss: 0.5720 Acc: 0.8262
Epoch 10/10
----------
train Loss: 0.5021 Acc: 0.8203
val Loss: 0.6098 Acc: 0.8262
Best val Acc: 0.840102
QAT para pesos de 8 bits e ativações de 4 bits;
Epoch 1/10
----------


OutOfMemoryError: CUDA out of memory. Tried to allocate 26.00 MiB (GPU 0; 3.81 GiB total capacity; 2.19 GiB already allocated; 9.00 MiB free; 2.27 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

Treino sem loop dando restart no kernel

In [14]:
print("QAT para pesos de 8 bits e ativações de 4 bits.")
model_name = 'qat_vgg16_w8_a4.pth'

quant_model = QuantVGG16(8,4)# definindo modelo
quant_model.load_state_dict(model.state_dict(), strict=False) # copiando pesos
quant_model.to(device) #movendo para a gpu

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(quant_model.parameters(),lr=0.0001)

quant_model = train_qat(quant_model,criterion,optimizer,train_loader,valid_loader,model_name)

QAT para pesos de 8 bits e ativações de 4 bits.
Epoch 1/10
----------


/usr/local/lib/python3.10/dist-packages/torch/_tensor.py:1255: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at ../c10/core/TensorImpl.h:1758.)
  return super(Tensor, self).rename(names)
/tmp/ipykernel_8148/2241107938.py:119: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:350.)
  x = torch.flatten(x, 1)


train Loss: 0.6718 Acc: 0.7611
val Loss: 0.6127 Acc: 0.8203
Epoch 2/10
----------
train Loss: 0.6439 Acc: 0.7674
val Loss: 0.5600 Acc: 0.8258
Epoch 3/10
----------
train Loss: 0.6442 Acc: 0.7681
val Loss: 0.6081 Acc: 0.8083
Epoch 4/10
----------
train Loss: 0.6310 Acc: 0.7723
val Loss: 0.6363 Acc: 0.8112
Epoch 5/10
----------
train Loss: 0.6258 Acc: 0.7729
val Loss: 0.6347 Acc: 0.7885
Epoch 6/10
----------
train Loss: 0.6235 Acc: 0.7744
val Loss: 0.6297 Acc: 0.8064
Epoch 7/10
----------
train Loss: 0.6090 Acc: 0.7798
val Loss: 0.6636 Acc: 0.7823
Epoch 8/10
----------
train Loss: 0.6024 Acc: 0.7811
val Loss: 0.6653 Acc: 0.7838
Epoch 9/10
----------
train Loss: 0.6018 Acc: 0.7823
val Loss: 0.6326 Acc: 0.8031
Epoch 10/10
----------
train Loss: 0.5796 Acc: 0.7848
val Loss: 0.6267 Acc: 0.8112
Best val Acc: 0.825832


In [15]:
print("QAT para pesos de 4 bits e ativações de 8 bits.")
model_name = 'qat_vgg16_w4_a8.pth'

quant_model = QuantVGG16(4,8)# definindo modelo
quant_model.load_state_dict(model.state_dict(), strict=False) # copiando pesos
quant_model.to(device) #movendo para a gpu

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(quant_model.parameters(),lr=0.0001)

quant_model = train_qat(quant_model,criterion,optimizer,train_loader,valid_loader,model_name)

QAT para pesos de 4 bits e ativações de 8 bits.
Epoch 1/10
----------


/usr/local/lib/python3.10/dist-packages/torch/_tensor.py:1255: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at ../c10/core/TensorImpl.h:1758.)
  return super(Tensor, self).rename(names)
/tmp/ipykernel_15180/2241107938.py:119: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:350.)
  x = torch.flatten(x, 1)


train Loss: 0.6057 Acc: 0.7912
val Loss: 0.5666 Acc: 0.8225
Epoch 2/10
----------
train Loss: 0.6110 Acc: 0.7839
val Loss: 0.6088 Acc: 0.8145
Epoch 3/10
----------
train Loss: 0.5770 Acc: 0.7930
val Loss: 0.5964 Acc: 0.8086
Epoch 4/10
----------
train Loss: 0.5690 Acc: 0.7985
val Loss: 0.6076 Acc: 0.8053
Epoch 5/10
----------
train Loss: 0.5705 Acc: 0.7962
val Loss: 0.6089 Acc: 0.8156
Epoch 6/10
----------
train Loss: 0.5658 Acc: 0.8010
val Loss: 0.5758 Acc: 0.8244
Epoch 7/10
----------
train Loss: 0.5533 Acc: 0.8054
val Loss: 0.6054 Acc: 0.8097
Epoch 8/10
----------
train Loss: 0.5471 Acc: 0.8022
val Loss: 0.6690 Acc: 0.7918
Epoch 9/10
----------
train Loss: 0.5260 Acc: 0.8161
val Loss: 0.5802 Acc: 0.8138
Epoch 10/10
----------
train Loss: 0.5119 Acc: 0.8187
val Loss: 0.6589 Acc: 0.8244
Best val Acc: 0.824369


In [15]:
print("QAT para pesos de 4 bits e ativações de 4 bits.")
model_name = 'qat_vgg16_w4_a4.pth'

quant_model = QuantVGG16(4,4)# definindo modelo
quant_model.load_state_dict(model.state_dict(), strict=False) # copiando pesos
quant_model.to(device) #movendo para a gpu

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(quant_model.parameters(),lr=0.0001)

quant_model = train_qat(quant_model,criterion,optimizer,train_loader,valid_loader,model_name)

QAT para pesos de 4 bits e ativações de 4 bits.
Epoch 1/10
----------


/usr/local/lib/python3.10/dist-packages/torch/_tensor.py:1255: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at ../c10/core/TensorImpl.h:1758.)
  return super(Tensor, self).rename(names)
/tmp/ipykernel_21800/2241107938.py:119: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:350.)
  x = torch.flatten(x, 1)


train Loss: 0.6847 Acc: 0.7498
val Loss: 0.6629 Acc: 0.7874
Epoch 2/10
----------
train Loss: 0.6594 Acc: 0.7605
val Loss: 0.6702 Acc: 0.7783
Epoch 3/10
----------
train Loss: 0.6593 Acc: 0.7624
val Loss: 0.6097 Acc: 0.8046
Epoch 4/10
----------
train Loss: 0.6498 Acc: 0.7707
val Loss: 0.7776 Acc: 0.7578
Epoch 5/10
----------
train Loss: 0.6472 Acc: 0.7669
val Loss: 0.6276 Acc: 0.8083
Epoch 6/10
----------
train Loss: 0.6441 Acc: 0.7663
val Loss: 0.6470 Acc: 0.7980
Epoch 7/10
----------
train Loss: 0.6356 Acc: 0.7709
val Loss: 0.6288 Acc: 0.7995
Epoch 8/10
----------
train Loss: 0.6329 Acc: 0.7678
val Loss: 0.6618 Acc: 0.8002
Epoch 9/10
----------
train Loss: 0.6182 Acc: 0.7778
val Loss: 0.5945 Acc: 0.8119
Epoch 10/10
----------
train Loss: 0.6018 Acc: 0.7823
val Loss: 0.6573 Acc: 0.7984
Best val Acc: 0.811928


## Avaliando os modelos quantizados

### W8A8

In [15]:
quant_model=QuantVGG16(8,8)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
quant_model.load_state_dict(torch.load("qat_vgg16_w8_a8.pth", map_location=device))
quant_model = quant_model.to(device)
quant_model.eval()

QuantVGG16(
  (features): Sequential(
    (0): QuantConv2d(
      3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)
      (input_quant): ActQuantProxyFromInjector(
        (_zero_hw_sentinel): StatelessBuffer()
      )
      (output_quant): ActQuantProxyFromInjector(
        (_zero_hw_sentinel): StatelessBuffer()
      )
      (weight_quant): WeightQuantProxyFromInjector(
        (_zero_hw_sentinel): StatelessBuffer()
        (tensor_quant): RescalingIntQuant(
          (int_quant): IntQuant(
            (float_to_int_impl): RoundSte()
            (tensor_clamp_impl): TensorClampSte()
            (delay_wrapper): DelayWrapper(
              (delay_impl): _NoDelay()
            )
          )
          (scaling_impl): StatsFromParameterScaling(
            (parameter_list_stats): _ParameterListStats(
              (first_tracked_param): _ViewParameterWrapper(
                (view_shape_impl): OverOutputChannelView(
                  (permute_impl): Identity()
                )
  

In [16]:
def calculate_metrics(loader, model, device):
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=[str(i) for i in range(6)], output_dict=True)
    kappa = cohen_kappa_score(all_labels, all_preds)
    
    per_class_accuracy = [report[str(i)]['precision'] for i in range(6)]
    mean_accuracy = sum(per_class_accuracy) / len(per_class_accuracy)

    return accuracy, mean_accuracy, kappa, classification_report(all_labels, all_preds, target_names=[str(i) for i in range(6)]), confusion_matrix(all_labels, all_preds)

In [17]:
def plot_confusion_matrix(cm, title, filename):
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[str(i) for i in range(6)], yticklabels=[str(i) for i in range(6)])
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title(title)
    plt.savefig(filename)
    plt.close()

In [22]:
warnings.filterwarnings("ignore", message="UndefinedMetricWarning")
print("Validation Set Metrics for w8a8 quantization:")
valid_accuracy, valid_mean_accuracy, valid_kappa, valid_report, valid_conf_matrix = calculate_metrics(valid_loader, quant_model, device)
print(f"Validation OA: {valid_accuracy:.4f}")
print(f"Validation AA: {valid_mean_accuracy:.4f}")
print(f"Validation Kappa: {valid_kappa:.4f}")
print("Validation Classification Report for w8a8 quantization:")
print(valid_report)
print("Validation Confusion Matrix for w8a8 quantization:")
print(valid_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de validação
plot_confusion_matrix(valid_conf_matrix, 'Validation Confusion Matrix', 'validation_confusion_matrix_vgg16_w8_a8.png')

Validation Set Metrics for w8a8 quantization:


/tmp/ipykernel_46149/2241107938.py:119: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:350.)
  x = torch.flatten(x, 1)


Validation OA: 0.8401
Validation AA: 0.7133
Validation Kappa: 0.7501
Validation Classification Report for w8a8 quantization:
              precision    recall  f1-score   support

           0       0.85      0.98      0.91      1253
           1       0.33      0.02      0.03       126
           2       0.81      0.83      0.82       895
           3       0.49      0.43      0.45        47
           4       0.83      0.69      0.75       182
           5       0.96      0.78      0.86       230

    accuracy                           0.84      2733
   macro avg       0.71      0.62      0.64      2733
weighted avg       0.82      0.84      0.82      2733

Validation Confusion Matrix for w8a8 quantization:
[[1227    1   25    0    0    0]
 [  74    2   49    0    1    0]
 [ 118    3  743   17   10    4]
 [   0    0   21   20    6    0]
 [   4    0   46    4  125    3]
 [  13    0   29    0    9  179]]


In [23]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="UndefinedMetricWarning")
print("Test Set Metrics for w8a8 quantization:")
test_accuracy, test_mean_accuracy, test_kappa, test_report, test_conf_matrix = calculate_metrics(test_loader, quant_model, device)
print(f"Test OA: {test_accuracy:.4f}")
print(f"Test AA: {test_mean_accuracy:.4f}")
print(f"Test Kappa: {test_kappa:.4f}")
print("Test Classification Report for w8a8 quantization:")
print(test_report)
print("Test Confusion Matrix for w8a8 quantization:")
print(test_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de teste
plot_confusion_matrix(test_conf_matrix, 'Test Confusion Matrix', 'test_confusion_matrix_vgg16_w8_a8.png')

Test Set Metrics for w8a8 quantization:
Test OA: 0.7549
Test AA: 0.6313
Test Kappa: 0.6104
Test Classification Report for w8a8 quantization:
              precision    recall  f1-score   support

           0       0.74      0.97      0.84      1880
           1       0.00      0.00      0.00       189
           2       0.75      0.59      0.66      1344
           3       0.58      0.21      0.31        71
           4       0.90      0.55      0.68       275
           5       0.82      0.91      0.87       346

    accuracy                           0.75      4105
   macro avg       0.63      0.54      0.56      4105
weighted avg       0.72      0.75      0.72      4105

Test Confusion Matrix for w8a8 quantization:
[[1831    0   49    0    0    0]
 [ 127    0   61    0    0    1]
 [ 508    0  788    5    4   39]
 [   2    0   51   15    3    0]
 [   6    0   86    6  150   27]
 [   4    0   17    0   10  315]]


### W8A4

In [25]:
quant_model=QuantVGG16(8,4)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
quant_model.load_state_dict(torch.load("qat_vgg16_w8_a4.pth", map_location=device))
quant_model = quant_model.to(device)
quant_model.eval()

QuantVGG16(
  (features): Sequential(
    (0): QuantConv2d(
      3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)
      (input_quant): ActQuantProxyFromInjector(
        (_zero_hw_sentinel): StatelessBuffer()
      )
      (output_quant): ActQuantProxyFromInjector(
        (_zero_hw_sentinel): StatelessBuffer()
      )
      (weight_quant): WeightQuantProxyFromInjector(
        (_zero_hw_sentinel): StatelessBuffer()
        (tensor_quant): RescalingIntQuant(
          (int_quant): IntQuant(
            (float_to_int_impl): RoundSte()
            (tensor_clamp_impl): TensorClampSte()
            (delay_wrapper): DelayWrapper(
              (delay_impl): _NoDelay()
            )
          )
          (scaling_impl): StatsFromParameterScaling(
            (parameter_list_stats): _ParameterListStats(
              (first_tracked_param): _ViewParameterWrapper(
                (view_shape_impl): OverOutputChannelView(
                  (permute_impl): Identity()
                )
  

In [26]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="UndefinedMetricWarning")
print("Validation Set Metrics for w8a4 quantization:")
valid_accuracy, valid_mean_accuracy, valid_kappa, valid_report, valid_conf_matrix = calculate_metrics(valid_loader, quant_model, device)
print(f"Validation OA: {valid_accuracy:.4f}")
print(f"Validation AA: {valid_mean_accuracy:.4f}")
print(f"Validation Kappa: {valid_kappa:.4f}")
print("Validation Classification Report for w8a4 quantization:")
print(valid_report)
print("Validation Confusion Matrix for w8a4 quantization:")
print(valid_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de validação
plot_confusion_matrix(valid_conf_matrix, 'Validation Confusion Matrix', 'validation_confusion_matrix_vgg16_w8_a4.png')

Validation Set Metrics for w8a4 quantization:
Validation OA: 0.8258
Validation AA: 0.6368
Validation Kappa: 0.7271
Validation Classification Report for w8a4 quantization:
              precision    recall  f1-score   support

           0       0.82      0.98      0.90      1253
           1       0.00      0.00      0.00       126
           2       0.82      0.78      0.80       895
           3       0.36      0.43      0.39        47
           4       0.85      0.66      0.75       182
           5       0.96      0.81      0.88       230

    accuracy                           0.83      2733
   macro avg       0.64      0.61      0.62      2733
weighted avg       0.79      0.83      0.80      2733

Validation Confusion Matrix for w8a4 quantization:
[[1228    0   25    0    0    0]
 [  92    0   34    0    0    0]
 [ 154    0  702   32    6    1]
 [   1    0   22   20    4    0]
 [   4    0   47    4  121    6]
 [  11    0   22    0   11  186]]


In [27]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="UndefinedMetricWarning")
print("Test Set Metrics for w8a4 quantization:")
test_accuracy, test_mean_accuracy, test_kappa, test_report, test_conf_matrix = calculate_metrics(test_loader, quant_model, device)
print(f"Test OA: {test_accuracy:.4f}")
print(f"Test AA: {test_mean_accuracy:.4f}")
print(f"Test Kappa: {test_kappa:.4f}")
print("Test Classification Report for w8a4 quantization:")
print(test_report)
print("Test Confusion Matrix for w8a4 quantization:")
print(test_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de teste
plot_confusion_matrix(test_conf_matrix, 'Test Confusion Matrix', 'test_confusion_matrix_vgg16_w8_a4.png')

Test Set Metrics for w8a4 quantization:
Test OA: 0.7164
Test AA: 0.5908
Test Kappa: 0.5463
Test Classification Report for w8a4 quantization:
              precision    recall  f1-score   support

           0       0.70      0.98      0.81      1880
           1       0.00      0.00      0.00       189
           2       0.74      0.47      0.57      1344
           3       0.47      0.27      0.34        71
           4       0.86      0.46      0.60       275
           5       0.78      0.93      0.85       346

    accuracy                           0.72      4105
   macro avg       0.59      0.52      0.53      4105
weighted avg       0.69      0.72      0.68      4105

Test Confusion Matrix for w8a4 quantization:
[[1845    0   35    0    0    0]
 [ 144    0   44    0    0    1]
 [ 643    0  629   14    5   53]
 [   5    0   42   19    5    0]
 [  12    0   93    7  126   37]
 [   3    0   10    0   11  322]]


### W4A8

In [28]:
quant_model=QuantVGG16(4,8)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
quant_model.load_state_dict(torch.load("qat_vgg16_w4_a8.pth", map_location=device))
quant_model = quant_model.to(device)
quant_model.eval()

QuantVGG16(
  (features): Sequential(
    (0): QuantConv2d(
      3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)
      (input_quant): ActQuantProxyFromInjector(
        (_zero_hw_sentinel): StatelessBuffer()
      )
      (output_quant): ActQuantProxyFromInjector(
        (_zero_hw_sentinel): StatelessBuffer()
      )
      (weight_quant): WeightQuantProxyFromInjector(
        (_zero_hw_sentinel): StatelessBuffer()
        (tensor_quant): RescalingIntQuant(
          (int_quant): IntQuant(
            (float_to_int_impl): RoundSte()
            (tensor_clamp_impl): TensorClampSte()
            (delay_wrapper): DelayWrapper(
              (delay_impl): _NoDelay()
            )
          )
          (scaling_impl): StatsFromParameterScaling(
            (parameter_list_stats): _ParameterListStats(
              (first_tracked_param): _ViewParameterWrapper(
                (view_shape_impl): OverOutputChannelView(
                  (permute_impl): Identity()
                )
  

In [29]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="UndefinedMetricWarning")
print("Validation Set Metrics for w4a8 quantization:")
valid_accuracy, valid_mean_accuracy, valid_kappa, valid_report, valid_conf_matrix = calculate_metrics(valid_loader, quant_model, device)
print(f"Validation OA: {valid_accuracy:.4f}")
print(f"Validation AA: {valid_mean_accuracy:.4f}")
print(f"Validation Kappa: {valid_kappa:.4f}")
print("Validation Classification Report for w4a8 quantization:")
print(valid_report)
print("Validation Confusion Matrix for w4a8 quantization:")
print(valid_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de validação
plot_confusion_matrix(valid_conf_matrix, 'Validation Confusion Matrix', 'validation_confusion_matrix_vgg16_w4_a8.png')

Validation Set Metrics for w4a8 quantization:
Validation OA: 0.8244
Validation AA: 0.6597
Validation Kappa: 0.7229
Validation Classification Report for w4a8 quantization:
              precision    recall  f1-score   support

           0       0.84      0.97      0.90      1253
           1       0.00      0.00      0.00       126
           2       0.78      0.84      0.81       895
           3       0.58      0.15      0.24        47
           4       0.79      0.63      0.70       182
           5       0.96      0.72      0.82       230

    accuracy                           0.82      2733
   macro avg       0.66      0.55      0.58      2733
weighted avg       0.79      0.82      0.80      2733

Validation Confusion Matrix for w4a8 quantization:
[[1218    0   35    0    0    0]
 [  80    0   44    0    2    0]
 [ 126    0  748    3   17    1]
 [   0    0   32    7    8    0]
 [   5    0   54    2  115    6]
 [  14    0   48    0    3  165]]


In [30]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="UndefinedMetricWarning")
print("Test Set Metrics for w8a4 quantization:")
test_accuracy, test_mean_accuracy, test_kappa, test_report, test_conf_matrix = calculate_metrics(test_loader, quant_model, device)
print(f"Test OA: {test_accuracy:.4f}")
print(f"Test AA: {test_mean_accuracy:.4f}")
print(f"Test Kappa: {test_kappa:.4f}")
print("Test Classification Report for w4a8 quantization:")
print(test_report)
print("Test Confusion Matrix for w4a8 quantization:")
print(test_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de teste
plot_confusion_matrix(test_conf_matrix, 'Test Confusion Matrix', 'test_confusion_matrix_vgg16_w4_a8.png')

Test Set Metrics for w8a4 quantization:
Test OA: 0.7325
Test AA: 0.6345
Test Kappa: 0.5717
Test Classification Report for w4a8 quantization:
              precision    recall  f1-score   support

           0       0.72      0.98      0.83      1880
           1       0.00      0.00      0.00       189
           2       0.72      0.54      0.61      1344
           3       0.71      0.07      0.13        71
           4       0.85      0.44      0.58       275
           5       0.80      0.92      0.86       346

    accuracy                           0.73      4105
   macro avg       0.63      0.49      0.50      4105
weighted avg       0.70      0.73      0.69      4105

Test Confusion Matrix for w4a8 quantization:
[[1840    0   40    0    0    0]
 [ 131    0   58    0    0    0]
 [ 561    0  723    1   12   47]
 [   3    0   58    5    5    0]
 [  11    0  110    1  122   31]
 [   3    0   22    0    4  317]]


### W4A4

In [31]:
quant_model=QuantVGG16(4,4)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
quant_model.load_state_dict(torch.load("qat_vgg16_w4_a4.pth", map_location=device))
quant_model = quant_model.to(device)
quant_model.eval()

QuantVGG16(
  (features): Sequential(
    (0): QuantConv2d(
      3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)
      (input_quant): ActQuantProxyFromInjector(
        (_zero_hw_sentinel): StatelessBuffer()
      )
      (output_quant): ActQuantProxyFromInjector(
        (_zero_hw_sentinel): StatelessBuffer()
      )
      (weight_quant): WeightQuantProxyFromInjector(
        (_zero_hw_sentinel): StatelessBuffer()
        (tensor_quant): RescalingIntQuant(
          (int_quant): IntQuant(
            (float_to_int_impl): RoundSte()
            (tensor_clamp_impl): TensorClampSte()
            (delay_wrapper): DelayWrapper(
              (delay_impl): _NoDelay()
            )
          )
          (scaling_impl): StatsFromParameterScaling(
            (parameter_list_stats): _ParameterListStats(
              (first_tracked_param): _ViewParameterWrapper(
                (view_shape_impl): OverOutputChannelView(
                  (permute_impl): Identity()
                )
  

In [32]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="UndefinedMetricWarning")
print("Validation Set Metrics for w4a4 quantization:")
valid_accuracy, valid_mean_accuracy, valid_kappa, valid_report, valid_conf_matrix = calculate_metrics(valid_loader, quant_model, device)
print(f"Validation OA: {valid_accuracy:.4f}")
print(f"Validation AA: {valid_mean_accuracy:.4f}")
print(f"Validation Kappa: {valid_kappa:.4f}")
print("Validation Classification Report for w4a4 quantization:")
print(valid_report)
print("Validation Confusion Matrix for w4a4 quantization:")
print(valid_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de validação
plot_confusion_matrix(valid_conf_matrix, 'Validation Confusion Matrix', 'validation_confusion_matrix_vgg16_w4_a4.png')

Validation Set Metrics for w4a4 quantization:
Validation OA: 0.8119
Validation AA: 0.6499
Validation Kappa: 0.7071
Validation Classification Report for w4a4 quantization:
              precision    recall  f1-score   support

           0       0.86      0.96      0.91      1253
           1       0.15      0.02      0.03       126
           2       0.77      0.81      0.79       895
           3       0.24      0.47      0.32        47
           4       0.91      0.52      0.66       182
           5       0.97      0.73      0.83       230

    accuracy                           0.81      2733
   macro avg       0.65      0.58      0.59      2733
weighted avg       0.80      0.81      0.80      2733

Validation Confusion Matrix for w4a4 quantization:
[[1208    7   38    0    0    0]
 [  79    2   43    0    1    1]
 [ 109    4  725   56    1    0]
 [   0    0   22   22    3    0]
 [   5    0   65   13   94    5]
 [   5    0   53    0    4  168]]


In [33]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="UndefinedMetricWarning")
print("Test Set Metrics for w4a4 quantization:")
test_accuracy, test_mean_accuracy, test_kappa, test_report, test_conf_matrix = calculate_metrics(test_loader, quant_model, device)
print(f"Test OA: {test_accuracy:.4f}")
print(f"Test AA: {test_mean_accuracy:.4f}")
print(f"Test Kappa: {test_kappa:.4f}")
print("Test Classification Report for w4a4 quantization:")
print(test_report)
print("Test Confusion Matrix for w4a4 quantization:")
print(test_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de teste
plot_confusion_matrix(test_conf_matrix, 'Test Confusion Matrix', 'test_confusion_matrix_vgg16_w4_a4.png')

Test Set Metrics for w4a4 quantization:
Test OA: 0.7306
Test AA: 0.5846
Test Kappa: 0.5705
Test Classification Report for w4a4 quantization:
              precision    recall  f1-score   support

           0       0.74      0.97      0.84      1880
           1       0.00      0.00      0.00       189
           2       0.70      0.56      0.62      1344
           3       0.37      0.28      0.32        71
           4       0.92      0.32      0.47       275
           5       0.78      0.90      0.84       346

    accuracy                           0.73      4105
   macro avg       0.58      0.51      0.52      4105
weighted avg       0.70      0.73      0.70      4105

Test Confusion Matrix for w4a4 quantization:
[[1828    0   52    0    0    0]
 [ 128    0   59    0    0    2]
 [ 520    1  751   24    0   48]
 [   3    0   47   20    1    0]
 [   6    0  136   10   87   36]
 [   1    0   25    0    7  313]]


# Exportando o modelo w4a4 para ONNX

In [34]:
quant_model.cpu()
quant_model.eval()

QuantVGG16(
  (features): Sequential(
    (0): QuantConv2d(
      3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)
      (input_quant): ActQuantProxyFromInjector(
        (_zero_hw_sentinel): StatelessBuffer()
      )
      (output_quant): ActQuantProxyFromInjector(
        (_zero_hw_sentinel): StatelessBuffer()
      )
      (weight_quant): WeightQuantProxyFromInjector(
        (_zero_hw_sentinel): StatelessBuffer()
        (tensor_quant): RescalingIntQuant(
          (int_quant): IntQuant(
            (float_to_int_impl): RoundSte()
            (tensor_clamp_impl): TensorClampSte()
            (delay_wrapper): DelayWrapper(
              (delay_impl): _NoDelay()
            )
          )
          (scaling_impl): StatsFromParameterScaling(
            (parameter_list_stats): _ParameterListStats(
              (first_tracked_param): _ViewParameterWrapper(
                (view_shape_impl): OverOutputChannelView(
                  (permute_impl): Identity()
                )
  

In [35]:
quant_model_export = quant_model

In [36]:
model_dir = os.environ['FINN_ROOT'] + "/notebooks/Classif-RetinopatiaDiabetica"
ready_model_filename = model_dir + "/classifier-vgg16-ready.onnx"

In [37]:
input_shape = (1,3,224,224)
input_a = np.random.randn(*input_shape).astype(np.float32)
input_t = torch.from_numpy(input_a)

export_qonnx(quant_model_export,export_path=ready_model_filename,input_t=input_t) 
qonnx_cleanup(ready_model_filename,out_file=ready_model_filename)

In [38]:
wrapped_model = ModelWrapper(ready_model_filename)

In [39]:
wrapped_model.set_tensor_datatype(wrapped_model.graph.input[0].name,DataType["FLOAT32"])

In [40]:
wrapped_model = wrapped_model.transform(ConvertQONNXtoFINN())

In [41]:
wrapped_model.save(ready_model_filename)
print(f"Model saved to {ready_model_filename}")

Model saved to /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/classifier-vgg16-ready.onnx


In [43]:
finnonnx_in_tensor_name = wrapped_model.graph.input[0].name
finnonnx_out_tensor_name = wrapped_model.graph.output[0].name
print("Input tensor name: %s" % finnonnx_in_tensor_name)
print("Output tensor name: %s" % finnonnx_out_tensor_name)
finnonnx_model_in_shape = wrapped_model.get_tensor_shape(finnonnx_in_tensor_name)
finnonnx_model_out_shape = wrapped_model.get_tensor_shape(finnonnx_out_tensor_name)
print("Input tensor shape: %s" % str(finnonnx_model_in_shape))
print("Output tensor shape: %s" % str(finnonnx_model_out_shape))
finnonnx_model_in_dt = wrapped_model.get_tensor_datatype(finnonnx_in_tensor_name)
finnonnx_model_out_dt = wrapped_model.get_tensor_datatype(finnonnx_out_tensor_name)
print("Input tensor datatype: %s" % str(finnonnx_model_in_dt.name))
print("Output tensor datatype: %s" % str(finnonnx_model_out_dt.name))
print("List of node operator types in the graph: ")
print([x.op_type for x in wrapped_model.graph.node])

Input tensor name: global_in
Output tensor name: global_out
Input tensor shape: [1, 3, 224, 224]
Output tensor shape: [1, 6]
Input tensor datatype: FLOAT32
Output tensor datatype: FLOAT32
List of node operator types in the graph: 
['Conv', 'Mul', 'Add', 'MultiThreshold', 'Mul', 'Conv', 'Mul', 'Add', 'MultiThreshold', 'Mul', 'MaxPool', 'Conv', 'Mul', 'Add', 'MultiThreshold', 'Mul', 'Conv', 'Mul', 'Add', 'MultiThreshold', 'Mul', 'MaxPool', 'Conv', 'Mul', 'Add', 'MultiThreshold', 'Mul', 'Conv', 'Mul', 'Add', 'MultiThreshold', 'Mul', 'Conv', 'Mul', 'Add', 'MultiThreshold', 'Mul', 'MaxPool', 'Conv', 'Mul', 'Add', 'MultiThreshold', 'Mul', 'Conv', 'Mul', 'Add', 'MultiThreshold', 'Mul', 'Conv', 'Mul', 'Add', 'MultiThreshold', 'Mul', 'MaxPool', 'Conv', 'Mul', 'Add', 'MultiThreshold', 'Mul', 'Conv', 'Mul', 'Add', 'MultiThreshold', 'Mul', 'Conv', 'Mul', 'Add', 'MultiThreshold', 'Mul', 'MaxPool', 'Flatten', 'MatMul', 'Mul', 'Add', 'MultiThreshold', 'Mul', 'MatMul', 'Mul', 'Add']


# Gerando Resultados estimados

In [96]:
model_dir = os.environ['FINN_ROOT'] + "/notebooks/Classif-RetinopatiaDiabetica"
ready_model_filename = model_dir + "/classifier-vgg16-ready.onnx"

In [97]:
wrapped_model = ModelWrapper(ready_model_filename)

In [8]:
model_dir = os.environ['FINN_ROOT'] + "/notebooks/Classif-RetinopatiaDiabetica"
output_dir=model_dir + "/generated_outputs_VGG16"
#Delete previous run results if exist
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
    print("Previous run results deleted!")
    
estimate = build_cfg.DataflowBuildConfig(
    steps=[
        "step_tidy_up",
        "step_streamline",
        "step_convert_to_hw",
        "step_create_dataflow_partition",
        "step_specialize_layers",
        "step_target_fps_parallelization",
        "step_apply_folding_config",
        "step_minimize_bit_width",
        "step_generate_estimate_reports",
],
    target_fps=1000000,
    output_dir=output_dir,
    synth_clk_period_ns=10.0,
    board="Pynq-Z1",
    shell_flow_type=build_cfg.ShellFlowType.VIVADO_ZYNQ,
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
    ],
)

Previous run results deleted!


In [9]:
%%time
build.build_dataflow_cfg(model_dir + "/classifier-vgg16-ready.onnx", estimate)

Building dataflow accelerator from /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/classifier-vgg16-ready.onnx
Intermediate outputs will be generated in /tmp/finn_dev_isadora
Final outputs will be generated in /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/generated_outputs_VGG16
Build log is at /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/generated_outputs_VGG16/build_dataflow.log
Running step: step_tidy_up [1/9]
Running step: step_streamline [2/9]
Running step: step_convert_to_hw [3/9]
Running step: step_create_dataflow_partition [4/9]
Running step: step_specialize_layers [5/9]
Running step: step_target_fps_parallelization [6/9]
Running step: step_apply_folding_config [7/9]
Running step: step_minimize_bit_width [8/9]
Running step: step_generate_estimate_reports [9/9]
Completed successfully
CPU times: user 7 s, sys: 3.66 s, total: 10.7 s
Wall time: 10.7 s


0

In [6]:
model_dir = os.environ['FINN_ROOT'] + "/notebooks/Classif-RetinopatiaDiabetica"
output_dir=model_dir + "/generated_outputs_VGG16"
#Delete previous run results if exist
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
    print("Previous run results deleted!")
    
estimate2 = build_cfg.DataflowBuildConfig(
    steps=[
        "step_tidy_up",
        "step_streamline",
        "step_convert_to_hw",
        "step_create_dataflow_partition",
        "step_specialize_layers",
        "step_target_fps_parallelization",
        "step_apply_folding_config",
        "step_minimize_bit_width",
        "step_generate_estimate_reports",
],
    target_fps=10,
    output_dir=output_dir,
    synth_clk_period_ns=10.0,
    board="Pynq-Z1",
    shell_flow_type=build_cfg.ShellFlowType.VIVADO_ZYNQ,
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
    ],
)

Previous run results deleted!


In [7]:
%%time
build.build_dataflow_cfg(model_dir + "/classifier-vgg16-ready.onnx", estimate2)

Building dataflow accelerator from /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/classifier-vgg16-ready.onnx
Intermediate outputs will be generated in /tmp/finn_dev_isadora
Final outputs will be generated in /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/generated_outputs_VGG16
Build log is at /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/generated_outputs_VGG16/build_dataflow.log
Running step: step_tidy_up [1/9]
Running step: step_streamline [2/9]
Running step: step_convert_to_hw [3/9]
Running step: step_create_dataflow_partition [4/9]
Running step: step_specialize_layers [5/9]
Running step: step_target_fps_parallelization [6/9]
Running step: step_apply_folding_config [7/9]
Running step: step_minimize_bit_width [8/9]
Running step: step_generate_estimate_reports [9/9]
Completed successfully
CPU times: user 7.05 s, sys: 3.72 s, total: 10.8 s
Wall time: 10.8 s


0